# Diffusion breakpoints and reversible state

Run all cells on CPU with `pip install -e '.[diffusion,notebooks]'`. The tiny native UNet is random: we inspect numerical state instead of presenting its 8×8 output as a semantic image result. For pretrained image behavior, use `experiments/flux2_writer/run.py`.

These commands use exactly the terminal CLI parser.

In [ ]:
import json
import shlex
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/saturn_pub").is_dir())
OUTPUT = ROOT / "outputs/notebooks"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Checkout:", ROOT.name)

In [ ]:
from saturn_pub.adapters.diffusion import DiffusionAdapter
from saturn_pub.cli import Debugger

dbg = Debugger(DiffusionAdapter.tiny().session(steps=4))
print(dbg.execute("break diffusion-step:1"))
stop = dbg.execute("continue")
print(json.dumps(stop, indent=2))
assert stop["frame"]["boundary"] == "diffusion-step:1"
dbg.execute("capture parent")

## Preview an intervention through the unchanged native consumer

Both futures consume their own changed or unchanged latent through the native UNet and scheduler. Inspecting state is distinct from measuring semantic image quality.

In [ ]:
for name in ("native", "candidate"):
    dbg.execute(f"fork {name} parent")
    dbg.execute(f"use {name}")
    if name == "candidate":
        dbg.execute("zero latent")
    dbg.execute("continue 3")
    print(
        name,
        dbg.session.inspect().boundary,
        float(dbg.session.read("latent").square().mean().sqrt()),
    )
    dbg.execute("use root")
dbg.execute("use native")
print("Equal final state:", dbg.execute("compare candidate")["equal_payload"])
dbg.execute("use root")
dbg.execute("commit candidate")
restore = dbg.execute("restore parent")
print("Exact restore:", restore["verified_exact"])
assert restore["verified_exact"]

## Bounded continue and durable load

`until BOUNDARY MAX_STEPS` ignores installed breakpoints and stops at the requested boundary or step limit. The boundary must be reachable within this trajectory.

In [ ]:
dbg.execute("delete all")
print(dbg.execute("until diffusion-step:3 2")["stop"])
dbg.execute("restore parent")
store = OUTPUT / "diffusion-debugger"
saved = dbg.execute(f"save {shlex.quote(str(store))} parent")["saved"]
other = Debugger(DiffusionAdapter.tiny().session(steps=4))
other.execute(f"load {shlex.quote(str(store))} {saved} imported")
assert other.execute("restore imported")["verified_exact"]
print(other.execute("where")["boundary"])